<a href="https://colab.research.google.com/github/clarachence/PLN-2026/blob/main/atividade_pratica08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício Prático Individual - Aula 08
Polaridade com negação em comentários sobre um condomínio (spaCy + léxico).

In [1]:
!pip install -q spacy pandas
!python -m spacy download pt_core_news_sm -q

import spacy
nlp = spacy.load("pt_core_news_sm")
print("Ambiente configurado com sucesso.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 93.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Ambiente configurado com sucesso.


In [2]:
corpus_exercicio = [
    "O síndico é muito prestativo e sempre resolve tudo rápido.",
    "A área de lazer não estava limpa quando visitei.",
    "O condomínio é caro, mas a estrutura compensa.",
    "Nunca vi um prédio tão mal administrado.",
]

lexico_positivo = {
    "ótimo", "excelente", "adorar", "maravilhoso", "perfeito", "recomendar",
    "satisfeito", "rápido", "eficiente", "atencioso", "gentil", "confiável",
    "espaçoso", "iluminado", "tranquilo", "seguro", "bom"
}
lexico_negativo = {
    "péssimo", "horrível", "detestar", "decepcionado", "lento", "descaso",
    "problema", "sujo", "escuro", "barulhento", "caro", "insatisfeito",
    "atrasado", "ruim", "complicado", "incomodar", "decepcionar"
}

NEGADORES = ("não", "nunca", "jamais")

def classificar(texto, pos_lex, neg_lex):
    doc = nlp(texto)
    p = n = 0
    for tok in doc:
        lema = tok.lemma_.lower()
        negado = any(f.lemma_.lower() in NEGADORES and f.dep_ == "advmod" for f in tok.children)
        if lema in pos_lex:
            if negado: n += 1
            else: p += 1
        elif lema in neg_lex:
            if negado: p += 1
            else: n += 1
    if p > n: return "Positivo", p, n
    if n > p: return "Negativo", p, n
    return "Neutro", p, n

## 1. Classificação de cada comentário

In [3]:
for i, t in enumerate(corpus_exercicio, 1):
    classificacao, pos, neg = classificar(t, lexico_positivo, lexico_negativo)
    print(f"Comentário {i}: {classificacao} (positivas={pos}, negativas={neg})")
    print(f"  \"{t}\"")

Comentário 1: Positivo (positivas=1, negativas=0)
  "O síndico é muito prestativo e sempre resolve tudo rápido."
Comentário 2: Neutro (positivas=0, negativas=0)
  "A área de lazer não estava limpa quando visitei."
Comentário 3: Negativo (positivas=0, negativas=1)
  "O condomínio é caro, mas a estrutura compensa."
Comentário 4: Neutro (positivas=0, negativas=0)
  "Nunca vi um prédio tão mal administrado."


## 2. Investigação dos resultados inesperados
Inspecionando lema, classe gramatical e dependência de cada token.

In [4]:
for i, t in enumerate(corpus_exercicio, 1):
    print(f"\nComentário {i}: {t}")
    for tok in nlp(t):
        print(f"  {tok.text:14} lema={tok.lemma_:14} pos={tok.pos_:6} dep={tok.dep_:10} head={tok.head.text}")


Comentário 1: O síndico é muito prestativo e sempre resolve tudo rápido.
  O              lema=o              pos=DET    dep=det        head=síndico
  síndico        lema=síndico        pos=NOUN   dep=nsubj      head=prestativo
  é              lema=ser            pos=AUX    dep=cop        head=prestativo
  muito          lema=muito          pos=ADV    dep=advmod     head=prestativo
  prestativo     lema=prestativo     pos=ADJ    dep=ROOT       head=prestativo
  e              lema=e              pos=CCONJ  dep=cc         head=resolve
  sempre         lema=sempre         pos=ADV    dep=advmod     head=resolve
  resolve        lema=resolver       pos=VERB   dep=conj       head=prestativo
  tudo           lema=tudo           pos=DET    dep=det        head=rápido
  rápido         lema=rápido         pos=NOUN   dep=xcomp      head=resolve
  .              lema=.              pos=PUNCT  dep=punct      head=prestativo

Comentário 2: A área de lazer não estava limpa quando visitei.
  A      

### Análise dos resultados
- **Comentário 1:** o resultado ficou certo (Positivo), mas só a palavra "rápido" foi contada. A palavra "prestativo" não estava no léxico. Faltava uma palavra.
- **Comentário 2:** o resultado ficou Neutro, mas a frase é negativa. A negação "não" foi percebida pelo código, mas a palavra "limpa" não estava no léxico, então não teve o que inverter. Além disso, o spaCy transformou "limpa" no lema `limpa` (e não `limpo`), por isso precisei colocar exatamente `limpa` no léxico.
- **Comentário 3:** o resultado ficou Negativo, mas a frase tem uma parte ruim ("caro") e uma boa ("a estrutura compensa"). A palavra "compensa" não foi contada porque o spaCy transformou ela em `compenso`, que não estava no léxico.
- **Comentário 4:** o resultado ficou Neutro, mas a frase é negativa. Aqui o problema **não** foi a negação: o "nunca" está ligado ao verbo "vi", e não à palavra "administrado". O que faltava era a palavra "mal" no léxico.

## 3. Ajuste do léxico (antes e depois)

In [5]:
pos_v2 = lexico_positivo | {"prestativo", "compenso", "limpa"}
neg_v2 = lexico_negativo | {"mal"}

for i, t in enumerate(corpus_exercicio, 1):
    print(f"Comentário {i}: {t}")
    print("  Antes :", classificar(t, lexico_positivo, lexico_negativo))
    print("  Depois:", classificar(t, pos_v2, neg_v2))

Comentário 1: O síndico é muito prestativo e sempre resolve tudo rápido.
  Antes : ('Positivo', 1, 0)
  Depois: ('Positivo', 2, 0)
Comentário 2: A área de lazer não estava limpa quando visitei.
  Antes : ('Neutro', 0, 0)
  Depois: ('Negativo', 0, 1)
Comentário 3: O condomínio é caro, mas a estrutura compensa.
  Antes : ('Negativo', 0, 1)
  Depois: ('Neutro', 1, 1)
Comentário 4: Nunca vi um prédio tão mal administrado.
  Antes : ('Neutro', 0, 0)
  Depois: ('Negativo', 0, 1)


Depois de adicionar as palavras que faltavam, os resultados ficaram assim:
- Comentário 1: Positivo
- Comentário 2: Negativo
- Comentário 3: Neutro (1 positiva e 1 negativa)
- Comentário 4: Negativo